# 🚑 Emergency Medical Supply Distribution Optimization

## Project Goal

This project aims to optimize the delivery of emergency medical supplies from a central hospital to multiple clinics.

The workflow consists of:

1. Predicting travel time using a trained XGBoost model
2. Using a Genetic Algorithm to find the optimal delivery sequence
3. Minimizing total travel time

This helps emergency response teams distribute medical supplies more efficiently during disasters and public health emergencies.

In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
import datetime
import random
import pickle
from copy import copy

In [3]:
# Load trained XGB model
loaded_model = pickle.load(
    open("C:/Projects/14_Ambulance_RL_System/model/model.pkl","rb")
)

print("Model loaded successfully")

Model loaded successfully


In [5]:
# Define Emergency Supply Locations
supply_locations = {
    "Central_Hospital": (40.7580, -73.9855),

    "Clinic_A": (40.7484, -73.9857),
    "Clinic_B": (40.7306, -73.9352),
    "Clinic_C": (40.7128, -74.0060),
    "Clinic_D": (40.7060, -74.0086),
    "Clinic_E": (40.7614, -73.9776),

    "Clinic_F": (40.7789, -73.9682),
    "Clinic_G": (40.7527, -73.9772),
    "Clinic_H": (40.7420, -74.0018),
    "Clinic_I": (40.7218, -73.9980),
    "Clinic_J": (40.7003, -73.9419),
    "Clinic_K": (40.6892, -74.0445)
}

In [6]:
START_LOCATION = "Central_Hospital"

clinic_locations = [
    location
    for location in supply_locations.keys()
    if location != START_LOCATION
]

print("Start Location:", START_LOCATION)
print("Clinic Locations:", clinic_locations)

Start Location: Central_Hospital
Clinic Locations: ['Clinic_A', 'Clinic_B', 'Clinic_C', 'Clinic_D', 'Clinic_E', 'Clinic_F', 'Clinic_G', 'Clinic_H', 'Clinic_I', 'Clinic_J', 'Clinic_K']


In [7]:
# Define Delivery Date
date_list = [4, 6, 2016]

year = date_list[2]
month = date_list[1]
day = date_list[0]

delivery_date = datetime.date(
    year,
    month,
    day
)

In [8]:
# Predict travel time between two locations
def travel_time_between_locations(
    point1,
    point2,
    hour=10,
    passenger_count=1,
    store_and_fwd_flag=0,
    pickup_minute=0
):
    # Simple Euclidean distance approximation
    trip_distance = np.linalg.norm(
    np.array(point2) - np.array(point1)
)
        
    model_data = {'passenger_count': passenger_count,
                  'pickup_longitude' : point1[1],
                  'pickup_latitude' : point1[0],
                  'dropoff_longitude' : point2[1],
                  'dropoff_latitude' : point2[0],
                  'store_and_fwd_flag' : store_and_fwd_flag,
                  'pickup_month' : delivery_date.month,
                  'pickup_day' : delivery_date.day,
                  'pickup_weekday' : delivery_date.weekday(),
                  'pickup_hour': hour,
                  'pickup_minute' : pickup_minute,
                  'latitude_difference' : point2[0] - point1[0],
                  'longitude_difference' : point2[1] - point1[1],
                  'trip_distance' : trip_distance,
                  'is_rush_hour': 1 if hour in [7,8,9,16,17,18] else 0,
                  'is_weekend': 1 if delivery_date.weekday() >= 5 else 0
                 }

    df = pd.DataFrame([model_data])

    pred = loaded_model.predict(df)
    
    return pred[0]

In [9]:
def create_guess(clinics, start_location=START_LOCATION):
    """
    Create one possible medical supply delivery route.
    The route always starts and ends at Central_Hospital.
    """

    clinic_order = copy(clinics)

    np.random.shuffle(clinic_order)

    route = [start_location] + clinic_order + [start_location]

    return route

In [10]:
create_guess(clinic_locations)

['Central_Hospital',
 'Clinic_B',
 'Clinic_I',
 'Clinic_A',
 'Clinic_E',
 'Clinic_D',
 'Clinic_F',
 'Clinic_J',
 'Clinic_G',
 'Clinic_H',
 'Clinic_C',
 'Clinic_K',
 'Central_Hospital']

In [11]:
# Create Initial Population
def create_generation(clinics, population=100):
    """
    Create many possible delivery routes.
    Every route starts and ends at Central_Hospital.
    """

    generation = [
        create_guess(clinics)
        for _ in range(population)
    ]

    return generation

In [12]:
# Fitness Function ( Lower travel time = Better Route)
coordinates = supply_locations

In [13]:
def fitness_score(route):

    score = 0

    for ix, point_id in enumerate(route[:-1]):

        score += travel_time_between_locations(
            coordinates[point_id],
            coordinates[route[ix+1]]
        )

    return score

In [14]:
route = create_guess(
    list(supply_locations.keys())
)

print(route)


['Central_Hospital', 'Clinic_B', 'Clinic_G', 'Clinic_A', 'Clinic_E', 'Clinic_H', 'Clinic_K', 'Clinic_C', 'Clinic_J', 'Central_Hospital', 'Clinic_F', 'Clinic_D', 'Clinic_I', 'Central_Hospital']


In [15]:
fitness_score(route)

np.float32(33.341274)

In [16]:
# Test the fitness function
route = create_guess(
    list(supply_locations.keys())
)
print(route)
print(
    "Total Route Time:",
    fitness_score(route)
)

['Central_Hospital', 'Clinic_C', 'Clinic_F', 'Clinic_A', 'Clinic_I', 'Clinic_K', 'Clinic_D', 'Central_Hospital', 'Clinic_G', 'Clinic_B', 'Clinic_E', 'Clinic_H', 'Clinic_J', 'Central_Hospital']
Total Route Time: 32.657795


In [17]:
# Create the initial population
test_generation = create_generation(
    clinic_locations,
    population=10
)

test_generation[:2]

[['Central_Hospital',
  'Clinic_A',
  'Clinic_H',
  'Clinic_I',
  'Clinic_E',
  'Clinic_C',
  'Clinic_D',
  'Clinic_K',
  'Clinic_B',
  'Clinic_G',
  'Clinic_J',
  'Clinic_F',
  'Central_Hospital'],
 ['Central_Hospital',
  'Clinic_K',
  'Clinic_F',
  'Clinic_E',
  'Clinic_I',
  'Clinic_D',
  'Clinic_G',
  'Clinic_A',
  'Clinic_B',
  'Clinic_J',
  'Clinic_C',
  'Clinic_H',
  'Central_Hospital']]

In [18]:
# Test check fitness
def check_fitness(routes):
    fitness_indicator = []
    for route in routes:
        fitness_indicator.append(
            (
            route,
            fitness_score(route)
            )
        )
    return fitness_indicator

results = check_fitness(test_generation)
results[:3]

[(['Central_Hospital',
   'Clinic_A',
   'Clinic_H',
   'Clinic_I',
   'Clinic_E',
   'Clinic_C',
   'Clinic_D',
   'Clinic_K',
   'Clinic_B',
   'Clinic_G',
   'Clinic_J',
   'Clinic_F',
   'Central_Hospital'],
  np.float32(30.086958)),
 (['Central_Hospital',
   'Clinic_K',
   'Clinic_F',
   'Clinic_E',
   'Clinic_I',
   'Clinic_D',
   'Clinic_G',
   'Clinic_A',
   'Clinic_B',
   'Clinic_J',
   'Clinic_C',
   'Clinic_H',
   'Central_Hospital'],
  np.float32(28.315527)),
 (['Central_Hospital',
   'Clinic_D',
   'Clinic_H',
   'Clinic_G',
   'Clinic_B',
   'Clinic_C',
   'Clinic_E',
   'Clinic_A',
   'Clinic_J',
   'Clinic_K',
   'Clinic_I',
   'Clinic_F',
   'Central_Hospital'],
  np.float32(31.530787))]

In [19]:
def get_breeders_from_generation(
    guesses,
    take_best_N=5,
    take_random_N=2
):

    fit_scores = check_fitness(guesses)

    sorted_guesses = sorted(
        fit_scores,
        key=lambda x: x[1]
    )

    new_generation = [
        x[0]
        for x in sorted_guesses[:take_best_N]
    ]

    best_guess = new_generation[0]

    for _ in range(take_random_N):
        ix = np.random.randint(len(guesses))
        new_generation.append(
            guesses[ix]
        )

    np.random.shuffle(new_generation)

    return new_generation, best_guess
breeders, best_route = get_breeders_from_generation(
    test_generation
)

print(best_route)


['Central_Hospital', 'Clinic_K', 'Clinic_F', 'Clinic_E', 'Clinic_I', 'Clinic_D', 'Clinic_G', 'Clinic_A', 'Clinic_B', 'Clinic_J', 'Clinic_C', 'Clinic_H', 'Central_Hospital']


In [20]:
def make_child(parent1, parent2):
    """
    Create a child route from two parent routes.
    Central_Hospital stays fixed at the start and end.
    Only clinic order is mixed.
    """

    parent1_middle = parent1[1:-1]
    parent2_middle = parent2[1:-1]

    child_middle = []

    half = len(parent1_middle) // 2

    child_middle.extend(parent1_middle[:half])

    for clinic in parent2_middle:
        if clinic not in child_middle:
            child_middle.append(clinic)

    child = [START_LOCATION] + child_middle + [START_LOCATION]

    return child

# Test
child = make_child(
    test_generation[0],
    test_generation[1]
)

child

['Central_Hospital',
 'Clinic_A',
 'Clinic_H',
 'Clinic_I',
 'Clinic_E',
 'Clinic_C',
 'Clinic_K',
 'Clinic_F',
 'Clinic_D',
 'Clinic_G',
 'Clinic_B',
 'Clinic_J',
 'Central_Hospital']

In [21]:
def make_children(
    old_generation,
    children_per_couple=2
):

    midpoint = len(old_generation) // 2

    next_generation = []

    for ix, parent in enumerate(old_generation[:midpoint]):

        for _ in range(children_per_couple):

            child = make_child(
                parent,
                old_generation[-ix - 1]
            )

            next_generation.append(child)

    return next_generation

In [22]:
# Test
children = make_children(
    breeders
)
print(len(children))

6


In [23]:
# Evolution Loop
fitness_history = []

current_generation = create_generation(
    clinic_locations,
    population=300
)

best_score_ever = float("inf")
best_route_ever = None

for generation in range(100):

    breeders, best_route = get_breeders_from_generation(
        current_generation,
        take_best_N=30,
        take_random_N=10
    )

    current_score = fitness_score(best_route)

    fitness_history.append(current_score)

    if current_score < best_score_ever:
        best_score_ever = current_score
        best_route_ever = best_route

    current_generation = make_children(
        breeders,
        children_per_couple=2
    )

    if generation % 10 == 0:

        print(f"Generation {generation}")

        print("Current Best Route:")
        print(best_route)

        print("Current Score:")
        print(current_score)

print("\n===== FINAL RESULT =====")

print("Best Route Ever:")
print(best_route_ever)

print("Best Score Ever:")
print(best_score_ever)

Generation 0
Current Best Route:
['Central_Hospital', 'Clinic_G', 'Clinic_I', 'Clinic_K', 'Clinic_B', 'Clinic_J', 'Clinic_F', 'Clinic_H', 'Clinic_D', 'Clinic_C', 'Clinic_A', 'Clinic_E', 'Central_Hospital']
Current Score:
27.905876
Generation 10
Current Best Route:
['Central_Hospital', 'Clinic_F', 'Clinic_E', 'Clinic_G', 'Clinic_A', 'Clinic_D', 'Clinic_H', 'Clinic_B', 'Clinic_J', 'Clinic_I', 'Clinic_C', 'Clinic_K', 'Central_Hospital']
Current Score:
26.898104
Generation 20
Current Best Route:
['Central_Hospital', 'Clinic_F', 'Clinic_E', 'Clinic_G', 'Clinic_A', 'Clinic_D', 'Clinic_H', 'Clinic_K', 'Clinic_I', 'Clinic_B', 'Clinic_J', 'Clinic_C', 'Central_Hospital']
Current Score:
27.538527
Generation 30
Current Best Route:
['Central_Hospital', 'Clinic_F', 'Clinic_E', 'Clinic_G', 'Clinic_A', 'Clinic_D', 'Clinic_H', 'Clinic_K', 'Clinic_I', 'Clinic_B', 'Clinic_J', 'Clinic_C', 'Central_Hospital']
Current Score:
27.538527
Generation 40
Current Best Route:
['Central_Hospital', 'Clinic_F', 'Clini

In [24]:
# Final result section
print("\n" + "="*50)
print("EMERGENCY MEDICAL SUPPLY OPTIMIZATION")
print("="*50)

print("\nBest Route:")

for stop in best_route_ever:
    print("→", stop)

print("\nPredicted Total Travel Time:")
print(f"{best_score_ever:.2f} minutes")


EMERGENCY MEDICAL SUPPLY OPTIMIZATION

Best Route:
→ Central_Hospital
→ Clinic_G
→ Clinic_F
→ Clinic_H
→ Clinic_E
→ Clinic_A
→ Clinic_B
→ Clinic_J
→ Clinic_I
→ Clinic_C
→ Clinic_D
→ Clinic_K
→ Central_Hospital

Predicted Total Travel Time:
26.83 minutes


In [25]:
# Save the corrected CSV files
route_df = pd.DataFrame({
    "Visit Order": range(1, len(best_route_ever) + 1),
    "Location": best_route_ever
})

route_df.to_csv(
    "optimized_supply_route.csv",
    index=False
)

print("optimized_supply_route.csv saved")

optimized_supply_route.csv saved


In [26]:
# Save fitness history
history_df = pd.DataFrame({
    "Generation": range(len(fitness_history)),
    "Score": fitness_history
})

history_df.to_csv(
    "fitness_history.csv",
    index=False
)

print("fitness_history.csv saved")

fitness_history.csv saved


In [27]:
# Save locations 
locations_df = pd.DataFrame(
    [
        [name, coord[0], coord[1]]
        for name, coord in supply_locations.items()
    ],
    columns=[
        "Location",
        "Latitude",
        "Longitude"
    ]
)

locations_df.to_csv(
    "locations.csv",
    index=False
)

print("locations.csv saved")

locations.csv saved
